In [73]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder,MinMaxScaler
from sklearn.metrics import r2_score, accuracy_score,mean_squared_error,median_absolute_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LinearRegression,LogisticRegression,Ridge,Lasso
from sklearn.svm import SVC,SVR
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [25]:
df = pd.read_csv(r"C:\Users\bunyo\Downloads\bmw_cars_market_dataset_synthetic.csv")

In [26]:
df.head()

,car_id,model,year,engine_size,horsepower,fuel_type,transmission,drivetrain,mileage_km,fuel_consumption_l_per_100km,co2_emissions_g_km,price_usd,doors,seats,body_type,color,owner_count,accident_history,service_history,country_sold
0,1,X5,2016,4.0,272,diesel,automatic,AWD,74655,8.9,196.0,66756,5,5,suv,white,2.0,no,partial,Netherlands
1,2,1 Series,2022,2.6,218,petrol,automatic,FWD,23469,8.4,190.0,26867,3,4,hatchback,black,1.0,no,partial,France
2,3,X1,2012,2.2,240,petrol,automatic,FWD,123273,7.8,174.0,31313,5,5,suv,white,3.0,no,full,Netherlands
3,4,X5,2022,3.8,316,diesel,NaN,AWD,33064,9.4,192.0,81594,5,5,suv,blue,1.0,no,NaN,France
4,5,7 Series,2023,3.1,294,petrol,automatic,RWD,23926,NaN,204.0,104105,4,5,sedan,white,1.0,no,full,Spain


# preprocessing

In [ ]:
df['Price_level'] = pd.cut(df['price_usd'], bins=[0, 34000, 55000, 226000  ], labels=['cheap', 'middle', 'expensive'])

In [39]:
df['Price_level'].value_counts()

Price_level
expensive    3529
cheap        3306
middle       3165
Name: count, dtype: int64

In [40]:
df = df.drop(columns=['car_id','model','color','country_sold'])

In [41]:
df = df.dropna()

In [43]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

In [44]:
for col in cat_cols:
    if df[col].nunique() < 3:
        dum = pd.get_dummies(df[col], prefix=col, dtype=int)
        df = df.drop(columns=[col])
        df = pd.concat([df,dum],axis=1)
    else:
        df[[col]] = OrdinalEncoder().fit_transform(df[[col]])
    

In [46]:
df.head()

,year,engine_size,horsepower,fuel_type,drivetrain,mileage_km,fuel_consumption_l_per_100km,co2_emissions_g_km,price_usd,doors,seats,body_type,owner_count,service_history,Price_level,transmission_automatic,transmission_manual,accident_history_no,accident_history_yes
0,2016,4.0,272,0.0,0.0,74655,8.9,196.0,66756,5,5,4.0,2.0,2.0,1.0,1,0,1,0
1,2022,2.6,218,3.0,1.0,23469,8.4,190.0,26867,3,4,2.0,1.0,2.0,0.0,1,0,1,0
2,2012,2.2,240,3.0,1.0,123273,7.8,174.0,31313,5,5,4.0,3.0,0.0,0.0,1,0,1,0
5,2010,1.9,190,0.0,0.0,182375,5.7,112.0,18615,5,5,4.0,5.0,2.0,0.0,1,0,1,0
6,2008,4.2,383,3.0,0.0,274131,10.7,222.0,27173,4,5,3.0,4.0,2.0,0.0,1,0,0,1


In [47]:
x = df.drop(columns=['price_usd','Price_level'])
y_reg = df['price_usd']
y_class = df['Price_level']

In [51]:
for i in x.columns:
    x[[i]] = MinMaxScaler().fit_transform(x[[i]])

In [53]:
x.head()

,year,engine_size,horsepower,fuel_type,drivetrain,mileage_km,fuel_consumption_l_per_100km,co2_emissions_g_km,doors,seats,body_type,owner_count,service_history,transmission_automatic,transmission_manual,accident_history_no,accident_history_yes
0,0.578947,0.80,0.262069,0.0,0.0,0.248850,0.689922,0.659933,1.000000,1.000000,1.00,0.4,1.0,1.0,0.0,1.0,0.0
1,0.894737,0.52,0.168966,1.0,0.5,0.078230,0.651163,0.639731,0.333333,0.666667,0.50,0.2,1.0,1.0,0.0,1.0,0.0
2,0.368421,0.44,0.206897,1.0,0.5,0.410910,0.604651,0.585859,1.000000,1.000000,1.00,0.6,0.0,1.0,0.0,1.0,0.0
5,0.263158,0.38,0.120690,0.0,0.0,0.607917,0.441860,0.377104,1.000000,1.000000,1.00,1.0,1.0,1.0,0.0,1.0,0.0
6,0.157895,0.84,0.453448,1.0,0.0,0.913770,0.829457,0.747475,0.666667,1.000000,0.75,0.8,1.0,1.0,0.0,0.0,1.0


# kerakli methodlar(Regression uchun)

In [62]:
from sklearn.ensemble import (
    BaggingRegressor, 
    GradientBoostingRegressor, 
    VotingRegressor, 
    StackingRegressor
)

In [58]:
x_train,x_test,y_train,y_test = train_test_split(x,y_reg, random_state=99,test_size=0.19)

# Ensemble Regression uchun

In [70]:
# bagging
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=110, 
    random_state=42
)

# boosting
boosting_model = GradientBoostingRegressor(
    n_estimators=50, 
    learning_rate=0.1, 
    random_state=42
)

#voting
voting = [
    ('lr', LinearRegression()),
    ('dt', DecisionTreeRegressor(max_depth=4)),
    ('gb', GradientBoostingRegressor(n_estimators=20))
]
voting_model = VotingRegressor(estimators=voting)


# stacking
stacking = [
    ('ridge', Ridge()),
    ('dt', DecisionTreeRegressor(max_depth=4))
]

stacking_model = StackingRegressor(
    estimators=stacking,
    final_estimator=LinearRegression()
)


# model 
modellar = {
    "Bagging": bagging_model,
    "Boosting": boosting_model,
    "Voting": voting_model,
    "Stacking": stacking_model
}



for nom, model in modellar.items():
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    
    r2 = r2_score(y_test, y_pred)
    mse = median_absolute_error(y_test, y_pred)
    
    print(f"{nom:10} | R2 Score: {r2:.3f} | MAE : {mse:.5f}")

Bagging    | R2 Score: 0.858 | MAE : 4972.58182
Boosting   | R2 Score: 0.824 | MAE : 7109.17793
Voting     | R2 Score: 0.747 | MAE : 8983.45898
Stacking   | R2 Score: 0.761 | MAE : 8566.87961


In [ ]:
# Bagging    | R2 Score: 0.857 | MAE : 5004.44000
# Boosting   | R2 Score: 0.824 | MAE : 7109.17793
# Voting     | R2 Score: 0.747 | MAE : 8983.45898
# Stacking   | R2 Score: 0.761 | MAE : 8566.87961

# Ensemble classification uchun

In [74]:
x_train,x_test,y_train,y_test = train_test_split(x,y_class, random_state=99,test_size=0.19)

In [67]:
from sklearn.ensemble import (
    BaggingClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

In [80]:
#bagging
bagging = BaggingClassifier(
    n_estimators= 100,
    estimator=DecisionTreeClassifier(),
    random_state=99
)


#boosting
boosting = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=99,
    max_depth=5
)


#voting
voting = [
    ('lr',LogisticRegression(max_iter=999)),
    ('rf',DecisionTreeClassifier(random_state=34,max_depth=6)),
    ('gr',GradientBoostingClassifier(n_estimators=46))
]
voting_model = VotingClassifier(estimators=voting)


#stacking
stacking = [
    ('l2',Lasso()),
    ('lr',LogisticRegression(max_iter=999)),
    ('dt',DecisionTreeClassifier(max_depth=5,random_state=55))
]
stacking_model = StackingClassifier(
    estimators=stacking,
    final_estimator=RandomForestClassifier(max_depth=5,random_state=99)
)

models = {
    'bagging':bagging,
    'boosting':boosting,
    'voting': voting_model,
    'stacking': stacking_model
}

for name,model in models.items():
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    
    accuracy = accuracy_score(y_test,y_pred)
    print(f"{name:10} | accuracy Score: {accuracy:.3f} |")

bagging    | accuracy Score: 0.812 |
boosting   | accuracy Score: 0.825 |
voting     | accuracy Score: 0.794 |
stacking   | accuracy Score: 0.787 |


# tabulate

In [92]:
from tabulate import tabulate

In [91]:
data = {
    'method':['r2 score', 'MEA','Accuracy'],
    'bagging':[0.857, 5004.44, 0.812],
    'boosting':[0.82, 7109.17, 0.825],
    'voting':[0.747, 8983.45,0.794],
    'stacking':[0.761, 8566.87, 0.787]
    
}
table = tabulate(data,headers='keys' ,tablefmt='fancy_grid')
print(table)

╒══════════╤═══════════╤════════════╤══════════╤════════════╕
│ method   │   bagging │   boosting │   voting │   stacking │
╞══════════╪═══════════╪════════════╪══════════╪════════════╡
│ r2 score │     0.857 │      0.82  │    0.747 │      0.761 │
├──────────┼───────────┼────────────┼──────────┼────────────┤
│ MEA      │  5004.44  │   7109.17  │ 8983.45  │   8566.87  │
├──────────┼───────────┼────────────┼──────────┼────────────┤
│ Accuracy │     0.812 │      0.825 │    0.794 │      0.787 │
╘══════════╧═══════════╧════════════╧══════════╧════════════╛
